In [0]:
%pip install databricks-feature-engineering xgboost --quiet

%restart_python

In [0]:
import xgboost as xgb

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import mlflow
import pyspark.sql.functions as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.tracking import MlflowClient

In [0]:
mlflow.set_registry_uri('databricks-uc')

In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
eval_split_table_name = "demand_eval_split"
feature_table_name = "daily_demand_features"
demand_labels_table_name = "daily_demand_labels"

RANDOM_STATE=  42

In [0]:
model = "daily_oil_gas_demand_xg_model"

In [0]:

model_name = f"{catalog_name}.{gold_schema_name}.{model}"


In [0]:
fe = FeatureEngineeringClient()

In [0]:
client  = MlflowClient()

In [0]:
def score_model_batch(
    catalog_name: str,
    gold_schema_name: str,
    demand_labels_table_name: str,
    model_name: str,
    model_version: int,
    fe: FeatureEngineeringClient,
# -> tuple[DataFrame, float]
) :
    """Load the test split from the labels table and score it with the model.

    Returns the joined (prediction, label, features) DataFrame and the test R2.
    """
    labels_df = spark.table(f"{catalog_name}.{gold_schema_name}.{demand_labels_table_name}")
    test_labels_df = labels_df.filter("split == 'test'")
    test_eval_df = test_labels_df.select("p_key", "transaction_date")

    predictions_df = fe.score_batch(
        model_uri=f"models:/{model_name}/{model_version}",
        df=test_eval_df,
    )

    prediction_with_label_df = predictions_df.select(
        "p_key", "transaction_date", "product_name", "destination_city", "prediction"
    ).join(
        test_labels_df.select("p_key", "transaction_date", "total_demand"),
        on=["p_key", "transaction_date"],
        how="inner",
    )

    n_expected = test_eval_df.count()
    n_actual = prediction_with_label_df.count()
    if n_actual != n_expected:
        raise ValueError(
            f"Row count mismatch after scoring/join: expected {n_expected}, got {n_actual}."
        )

    pdf = prediction_with_label_df.select("total_demand", "prediction").toPandas()
    test_r2 = r2_score(pdf["total_demand"], pdf["prediction"])

    return prediction_with_label_df, test_r2

In [0]:
versions = client.search_model_versions(f"name='{model_name}'")

In [0]:
def get_metric(v, key="val_r2"):
    run = client.get_run(v.run_id)
    return run.data.metrics.get(key, float("-inf"))

best = max(versions, key=get_metric)
print(best.version, get_metric(best))

In [0]:
model_uri = f"models:/{model_name}/{best.version}"

In [0]:
challenger_predictions_df, challenger_test_r2 = score_model_batch(
    catalog_name=catalog_name,
    gold_schema_name=gold_schema_name,
    demand_labels_table_name=demand_labels_table_name,
    model_name=model_name,
    model_version=best.version,
    fe=fe,
)

print(f"Test R2: {challenger_test_r2}")
# display(predictions_df)

In [0]:
challenger_predictions_df.show()

In [0]:
challenger_predictions_df.write.format("delta").mode("overwrite").saveAsTable(
        f"{catalog_name}.{gold_schema_name}.predictions_with_labels_{model}_{best.version}")

In [0]:
if challenger_test_r2 < 0.8:
    print("Model failed test R2 threshold")
else:
    print("Model passed test R2 threshold")

    client.set_model_version_tag(
        name=model_name,
        version=best.version,
        key="test_r2",
        value=f"{challenger_test_r2:.4f}",
    )
    print('register model as Challenger!')
    client.set_registered_model_alias(
        name=model_name,
        alias="Challenger",
        version=best.version
    )